# Treinamentos controlados D-P1 e D-P2

Este notebook mede o efeito isolado de dois pré-processamentos sobre a DeepResUNet central. D-P1 usa Shades-of-Gray e D-P2 usa CLAHE em L\*. Arquitetura, split, seed, resolução, loss, treinamento e avaliação são iguais ao D0. Teste interno e FUSeg permanecem fechados.

## Contraste científico

- `D-P1 − D0`: efeito de constância de cor;
- `D-P2 − D0`: efeito de contraste local de luminosidade.

D-P1 e D-P2 começam da mesma seed 42; nenhum checkpoint do D0 é carregado. Os métodos não são combinados.

## Organização prática

Cada treino pode levar tempo semelhante ao D0. Execute P1 e P2 em células separadas. Se o Colab desconectar, use o diretório daquela execução em `P1_RESUME_DIR` ou `P2_RESUME_DIR`; não crie uma execução nova para continuar.

## 1. Montar Drive, localizar projeto e registrar `src`

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, yaml

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Execução fora do Colab.")

candidates = [
    Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas"),
    Path("/content/drive/MyDrive/Projeto_TCC_Lucas"),
]
PROJECT_ROOT = next((p for p in candidates if p.is_dir()), Path.cwd())
SOURCE_ROOT = PROJECT_ROOT / "src"
os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(SOURCE_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
print("Projeto:", PROJECT_ROOT)
print("Código-fonte:", SOURCE_ROOT)

## 2. Instalar dependências registradas

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

## 3. Copiar dados para o cache temporário

In [ ]:
DATA_ROOT = Path("/content/tcc_data") if Path("/content").is_dir() else PROJECT_ROOT
if DATA_ROOT != PROJECT_ROOT:
    shutil.copytree(PROJECT_ROOT / "Dataset", DATA_ROOT / "Dataset", dirs_exist_ok=True)
print("Dados:", DATA_ROOT / "Dataset")

## 4. Carregar e comparar as configurações

In [ ]:
from IPython.display import display

D0_CONFIG = yaml.safe_load((PROJECT_ROOT / "configs/baseline_d0.yaml").read_text(encoding="utf-8"))
P1_CONFIG_PATH = PROJECT_ROOT / "configs/preprocessing_dp1_shades_of_gray.yaml"
P2_CONFIG_PATH = PROJECT_ROOT / "configs/preprocessing_dp2_clahe_lstar.yaml"
P1_CONFIG = yaml.safe_load(P1_CONFIG_PATH.read_text(encoding="utf-8"))
P2_CONFIG = yaml.safe_load(P2_CONFIG_PATH.read_text(encoding="utf-8"))
display(
    {
        "D0": D0_CONFIG["data"]["color_preprocessing"],
        "D-P1": P1_CONFIG["data"]["color_preprocessing"],
        "D-P2": P2_CONFIG["data"]["color_preprocessing"],
    }
)
for candidate in (P1_CONFIG, P2_CONFIG):
    assert candidate["model"] == D0_CONFIG["model"]
    assert candidate["loss"] == D0_CONFIG["loss"]
    assert candidate["training"] == D0_CONFIG["training"]
    assert candidate["threshold_selection"] == D0_CONFIG["threshold_selection"]
    assert candidate["seed"] == D0_CONFIG["seed"] == 42
print("Controle aprovado: somente pré-processamento e identificação diferem.")

## 5. Executar os testes de configuração e transforms

In [ ]:
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = (
    str(SOURCE_ROOT) + os.pathsep + test_environment.get("PYTHONPATH", "")
)
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_preprocessing.py",
        "tests/test_preprocessing_training_config.py",
    ],
    cwd=PROJECT_ROOT,
    env=test_environment,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

## 6. Visualizar a entrada real da rede

Esta célula confirma P0/P1/P2 após resize. A máscara deve permanecer idêntica entre os braços; apenas a imagem muda.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tcc_segmentation.data import DevelopmentDataset, ResizePad
from tcc_segmentation.preprocessing_experiment import make_preprocessing_pair_transform

manifest = PROJECT_ROOT / P1_CONFIG["data"]["validation_manifest"]
datasets = {
    "P0": DevelopmentDataset(manifest, DATA_ROOT, "validation", transform=ResizePad(384, 512)),
    "P1": DevelopmentDataset(
        manifest, DATA_ROOT, "validation", transform=make_preprocessing_pair_transform(P1_CONFIG)
    ),
    "P2": DevelopmentDataset(
        manifest, DATA_ROOT, "validation", transform=make_preprocessing_pair_transform(P2_CONFIG)
    ),
}
sample_id = P1_CONFIG["visualization"]["fixed_validation_ids"][2]
index = next(i for i, row in enumerate(datasets["P0"].rows) if row["canonical_id"] == sample_id)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
masks = []
for column, key in enumerate(("P0", "P1", "P2")):
    sample = datasets[key][index]
    masks.append(sample["mask"])
    axes[column].imshow(sample["image"].transpose(1, 2, 0))
    axes[column].set_title(key)
    axes[column].axis("off")
axes[3].imshow(masks[0][0], cmap="gray")
axes[3].set_title("máscara comum")
axes[3].axis("off")
assert all(np.array_equal(masks[0], mask) for mask in masks[1:])
plt.tight_layout()
plt.show()

## 7. Função de execução com saída contínua

Ela imprime cada época e preserva o erro original caso algo falhe.

In [ ]:
def run_stream(command):
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    returncode = process.wait()
    print("\nreturncode=", returncode)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode

# Parte A — D-P1: Shades-of-Gray

Execute esta parte até o fim. Para uma execução nova, mantenha `P1_RESUME_DIR = None`. Em caso de desconexão, informe o diretório criado.

In [ ]:
# Exemplo: P1_RESUME_DIR = PROJECT_ROOT / 'results/day6/DP1/DP1_seed42_...'
P1_RESUME_DIR = None
p1_command = [
    sys.executable,
    "scripts/run_preprocessing_experiment.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--config",
    str(P1_CONFIG_PATH),
]
if P1_RESUME_DIR is not None:
    p1_command += ["--resume-dir", str(P1_RESUME_DIR)]
run_stream(p1_command)

## 8. Visualizar o resultado D-P1

In [ ]:
from PIL import Image
import pandas as pd

P1_OUTPUT = (
    Path(P1_RESUME_DIR)
    if P1_RESUME_DIR
    else max(
        (PROJECT_ROOT / P1_CONFIG["runtime"]["output_root"]).iterdir(),
        key=lambda p: p.stat().st_mtime,
    )
)
P1_SUMMARY = json.loads((P1_OUTPUT / "final_summary.json").read_text())
display(
    {
        "run": P1_SUMMARY["run_id"],
        "melhor_época": P1_SUMMARY["training"]["best_epoch"],
        "Dice_0.5": P1_SUMMARY["selection"]["metrics_at_0_5"]["macro"]["dice"]["mean"],
        "threshold": P1_SUMMARY["selection"]["threshold"],
        "Dice_selecionado": P1_SUMMARY["selection"]["metrics_selected"]["macro"]["dice"]["mean"],
    }
)
display(Image.open(P1_OUTPUT / "training_curves.png"))
display(Image.open(P1_OUTPUT / "threshold_curves.png"))
display(Image.open(P1_OUTPUT / "fixed_validation_panel.png"))

# Parte B — D-P2: CLAHE em L\*

Execute esta parte separadamente. Ela não carrega pesos nem resultados de P1.

In [ ]:
# Exemplo: P2_RESUME_DIR = PROJECT_ROOT / 'results/day6/DP2/DP2_seed42_...'
P2_RESUME_DIR = None
p2_command = [
    sys.executable,
    "scripts/run_preprocessing_experiment.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--config",
    str(P2_CONFIG_PATH),
]
if P2_RESUME_DIR is not None:
    p2_command += ["--resume-dir", str(P2_RESUME_DIR)]
run_stream(p2_command)

## 9. Visualizar o resultado D-P2

In [ ]:
P2_OUTPUT = (
    Path(P2_RESUME_DIR)
    if P2_RESUME_DIR
    else max(
        (PROJECT_ROOT / P2_CONFIG["runtime"]["output_root"]).iterdir(),
        key=lambda p: p.stat().st_mtime,
    )
)
P2_SUMMARY = json.loads((P2_OUTPUT / "final_summary.json").read_text())
display(
    {
        "run": P2_SUMMARY["run_id"],
        "melhor_época": P2_SUMMARY["training"]["best_epoch"],
        "Dice_0.5": P2_SUMMARY["selection"]["metrics_at_0_5"]["macro"]["dice"]["mean"],
        "threshold": P2_SUMMARY["selection"]["threshold"],
        "Dice_selecionado": P2_SUMMARY["selection"]["metrics_selected"]["macro"]["dice"]["mean"],
    }
)
display(Image.open(P2_OUTPUT / "training_curves.png"))
display(Image.open(P2_OUTPUT / "threshold_curves.png"))
display(Image.open(P2_OUTPUT / "fixed_validation_panel.png"))

## 10. Comparação provisória — não abrir teste

A tabela abaixo organiza D0/P1/P2 usando somente validação. Não escolha manualmente outra época ou threshold depois de vê-la.

In [ ]:
D0_DIR = PROJECT_ROOT / "results/day5/D0/D0_seed42_20260829T105601Z"
D0_SUMMARY = json.loads((D0_DIR / "final_summary.json").read_text())
rows = []
for label, summary in [("D0/P0", D0_SUMMARY), ("D-P1", P1_SUMMARY), ("D-P2", P2_SUMMARY)]:
    sel = summary["selection"]
    rows.append(
        {
            "braço": label,
            "melhor_época": summary["training"]["best_epoch"],
            "Dice_0.5": sel["metrics_at_0_5"]["macro"]["dice"]["mean"],
            "threshold": sel["threshold"],
            "Dice_selecionado": sel["metrics_selected"]["macro"]["dice"]["mean"],
            "IoU_selecionado": sel["metrics_selected"]["macro"]["iou"]["mean"],
            "precisão": sel["metrics_selected"]["macro"]["precision"]["mean"],
            "recall": sel["metrics_selected"]["macro"]["recall"]["mean"],
        }
    )
display(pd.DataFrame(rows))

## Artefatos da execução

Os arquivos produzidos por configuração são `final_summary.json`, `history.csv`, `training_curves.png`, `threshold_curve.csv`, `threshold_curves.png`, `threshold_selection.json`, `fixed_validation_panel.png` e `validation_by_area_bin.csv`.

A comparação precisa ser revisada antes do augmentation. Não execute teste interno, FUSeg, P1+P2 ou qualquer ajuste adicional.